# 📝 LangChain 에이전트와 도구 과제 LV3(통합) — 리뷰 인텔리전스·검색기 평가

> 두 갈래를 통합합니다. 1번은 흩어진 리뷰를 **구조화 수집 → 집계 → 인사이트**로 잇고, 2번은 **모델을 부르지 않고** 검색기를 재서 실제로 쓸 `k` 를 근거를 가지고 고릅니다.

## 풀이 방법
1. 맨 위 **준비 셀들**을 위에서부터 실행하세요. `.env` 에 본인 **`OPENAI_API_KEY`** 가 필요합니다(1번에서만 씁니다 — 2번은 모델을 부르지 않습니다).
2. 각 문제는 **여러 단계**로 나뉩니다. 단계마다 할 일이 셀에 적혀 있으니 순서대로 채우세요.
3. 1번은 모델이 만드는 답이 **실행할 때마다 달라져** 채점을 **타입·구조**로 합니다. 2번은 모델을 부르지 않아 **값이 언제나 같으므로** 실측값과 대조해 채점합니다.

화이팅!

아래 준비 셀을 먼저 실행하세요. 2번에서 쓸 색인은 그 문제 바로 앞에 따로 준비 셀이 있습니다.

In [ ]:
# [제공 코드] OpenAI 키 준비 — 이 셀은 실행만 하세요.
# 14~16일차와 같은 방식입니다: .env 파일에 넣어 둔 OPENAI_API_KEY 를 읽어 옵니다.
import os

from dotenv import load_dotenv

load_dotenv(".env")       # 같은 폴더의 .env
load_dotenv("../.env")    # 정답 폴더에서 실행하는 경우

# 키를 먼저 확인합니다 — 모델을 만든 뒤에 검사하면 인증 오류가 먼저 나서 이 안내가 묻힙니다.
if not os.getenv("OPENAI_API_KEY"):
    raise RuntimeError(
        "이 노트북은 실제 OpenAI 호출이 필요합니다 — OPENAI_API_KEY 를 찾지 못했습니다.\n"
        "  1) 일차 폴더에서  cp .env.example .env\n"
        "  2) .env 를 열어 본인 키를 채우세요\n"
        "  3) 커널을 재시작한 뒤 이 셀부터 다시 실행하세요")

print("OpenAI 키 확인 완료 — 이제 LangChain 으로 모델을 만들 수 있습니다.")

In [ ]:
# [제공 코드] 에이전트 공통 준비
from langchain.agents import create_agent
from langchain_core.messages import ToolMessage

from langchain_openai import ChatOpenAI

# temperature=0 : 같은 질문에 되도록 일정한 답을 받는 설정(수업·채점용).
model = ChatOpenAI(model='gpt-4o-mini', temperature=0)

## 1. 리뷰 인텔리전스 파이프라인
**배경**: 흩어진 도서 리뷰(**비정형**)를 측면별 감성으로 **구조화 수집**한 뒤, `pandas` 로 **집계**해 측면별 인사이트를 뽑는 파이프라인을 만듭니다. "비정형 → 정형 → 인사이트"의 실무 서사입니다.

제공 셀의 `ReviewBatch` 스키마와 지시문·리뷰 묶음을 씁니다.

In [ ]:
# [제공 코드] 측면별 감성 스키마 — 이 셀은 실행만 하세요.
from typing import Literal

from pydantic import BaseModel, Field


class AspectOpinion(BaseModel):
    aspect: Literal["배송", "품질", "가격", "내용"] = Field(description="리뷰가 언급한 측면")
    sentiment: Literal["긍정", "부정", "중립"] = Field(description="그 측면에 대한 감성")
    evidence: str = Field(description="그렇게 판단한 근거가 된 리뷰 속 표현")


class ReviewAspects(BaseModel):
    review_id: str = Field(description="리뷰 번호")
    opinions: list[AspectOpinion] = Field(description="이 리뷰가 다룬 측면별 의견 목록")


class ReviewBatch(BaseModel):
    results: list[ReviewAspects] = Field(description="여러 리뷰의 측면별 분석 결과 목록")


print("스키마 준비 완료")

In [ ]:
# [제공 코드] 리뷰 전체 로드 — 이 셀은 실행만 하세요.
import csv

_rows = list(csv.DictReader(open('data/bookstore_reviews.csv', encoding='utf-8')))
REVIEW_INSTR = (
    '다음 도서 리뷰들을 측면별로 분석하세요. 각 리뷰에서 언급된 측면(배송·품질·가격·내용)마다 감성(긍정·부정·중립)과 근거 표현을 뽑아 주세요. 언급되지 않은 측면은 포함하지 마세요.\n\n'
)
REVIEW_BLOCK = '\n'.join(f"[{r['review_id']}] {r['text']}" for r in _rows)
print('리뷰', len(_rows), '건 준비')

### 1단계 — 측면 감성 구조화 수집
`model.with_structured_output(ReviewBatch)` 로 **`REVIEW_INSTR + REVIEW_BLOCK`** 을 `invoke` 해 결과를 변수 **`batch`** 에 담으세요. 이어서 `(리뷰번호, 측면, 감성)` 튜플의 리스트 **`records`** 로 펼치세요.

리뷰 한 건이 여러 측면을 말할 수 있으니, `records` 의 길이는 리뷰 수보다 많아집니다. 순서는 **결과에 담긴 리뷰 순서, 그 안에서는 의견 순서** 그대로입니다.

<details><summary>힌트</summary>

```text
접근방법:
- 모델에 스키마를 씌워 한 번 부르고, 결과를 두 겹 반복으로 납작하게 편다.

세부구현:
1. 지시문과 리뷰 묶음을 이어 붙인 하나의 문자열을 스키마 씌운 모델에 넘긴다.
2. 결과의 리뷰 목록을 돌고, 그 안에서 측면별 의견 목록을 다시 돈다.
3. 리뷰 번호·측면·감성 세 값을 튜플 하나로 만들어 차례로 모은다.
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert isinstance(batch.results, list) and len(batch.results) >= 1
assert isinstance(records, list) and len(records) >= 1   # 측면 의견들이 펼쳐졌다
assert all(len(r) == 3 for r in records)                 # (리뷰번호, 측면, 감성) 세 칸
assert all(a in {'배송', '품질', '가격', '내용'} for _, a, _ in records)
assert all(s in {'긍정', '부정', '중립'} for _, _, s in records)
# 위 조건은 손으로 지어낸 튜플도 만족한다 — records 가 batch 를 펼친 것인지 그대로 대조한다
assert records == [(ra.review_id, o.aspect, o.sentiment)
                   for ra in batch.results for o in ra.opinions], \
    'batch 를 순서대로 펼친 결과여야 합니다'
print('✅ 통과!')

### 2단계 — pandas 집계
`records` 로 `DataFrame` 을 만들어 변수 **`df`** 에 담고(열: `review_id`, `aspect`, `sentiment`), **측면별 부정 건수**를 세어 변수 **`neg_by_aspect`**(딕셔너리: 측면→부정 건수)에 담으세요.

부정이 한 건도 없는 측면은 **열쇠 자체가 없어야** 합니다(0 으로 채워 넣지 않습니다).

<details><summary>힌트</summary>

```text
접근방법:
- 튜플 목록으로 표를 만들고, 감성이 부정인 행만 남겨 측면별로 센다.

세부구현:
1. 튜플 리스트로 DataFrame 을 만들되 열 이름을 직접 지정한다.
2. 감성 열이 부정인 행만 걸러 낸다.
3. 그 행들의 측면 열을 값별로 세어 딕셔너리로 바꾼다.
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert list(df.columns) == ['review_id', 'aspect', 'sentiment']
assert len(df) == len(records)
assert isinstance(neg_by_aspect, dict)
assert set(neg_by_aspect) <= {'배송', '품질', '가격', '내용'}
assert all(int(v) >= 0 for v in neg_by_aspect.values())   # 건수(정수)
# 여기까지는 빈 딕셔너리도 통과한다 — df 의 부정 행과 실제로 대조한다
_neg_rows = df[df['sentiment'] == '부정']
assert set(neg_by_aspect) == set(_neg_rows['aspect']), '부정이 나온 측면이 빠졌거나 더 들어 있습니다'
assert sum(int(v) for v in neg_by_aspect.values()) == len(_neg_rows), '부정 건수의 합이 df 의 부정 행 수와 다릅니다'
print('✅ 통과!')

### 3단계 — 측면별 요약
측면별 **긍정·부정 건수**를 한눈에 보이게 정리합니다. `df` 를 측면×감성으로 **교차집계**해 변수 **`summary`** 에 담으세요(`pivot_table`, 값이 없으면 0). 행(index)은 **측면**, 열(columns)은 **감성**입니다. 어느 측면이 가장 **부정적**인지 눈으로 확인하세요.

<details><summary>힌트</summary>

```text
접근방법:
- 앞 단계에서 만든 표를 측면(행)과 감성(열)으로 교차집계해 건수를 센다.

세부구현:
1. pivot_table 에 행 축과 열 축을 지정한다.
2. 세는 것이 목적이므로 집계 함수는 개수를 세는 것으로 하고, 값 열은 아무 열이나 준다.
3. 한 번도 나오지 않은 조합이 비어 있지 않도록 채울 값을 0 으로 지정한다.
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert summary.index.name == 'aspect'
assert set(summary.index) <= {'배송', '품질', '가격', '내용'}
assert set(summary.columns) <= {'긍정', '부정', '중립'}
assert int(summary.to_numpy().sum()) == len(df)   # 교차집계는 전체 의견 수를 나눠 담는다
print('✅ 통과!')

## 2. 검색기를 재서 `k` 를 정하기
**배경**: 1번은 모델이 판단하는 문제였습니다. 2번은 **모델을 한 번도 부르지 않습니다.** 서점 고객센터 FAQ 22건을 색인하고, 손으로 만든 평가셋 14문항으로 검색을 **재서**, 실제 서비스에 넣을 `k` 를 **근거를 가지고** 고릅니다.

지금까지는 `k=2` 나 `k=3` 을 그냥 썼습니다. 이번에는 **왜 그 값인지**를 수치로 말할 수 있게 만듭니다.

제공 셀이 색인(`bs_store`)과 검색 함수(`search_ids`), 평가셋(`bs_eval`)을 만들어 둡니다. **지표 네 개는 LV1 9~12번에서 만든 것을 그대로** 다시 정의해 쓰세요.

In [ ]:
# [제공 코드] 서점 FAQ 색인과 평가셋 — 이 셀은 실행만 하세요(임베딩에 잠시 걸립니다).
#  FAQ 한 행이 조각 하나입니다(한 건이 짧아 자를 것이 없습니다) -> 조각 id 는 FAQ 의 id 그대로입니다.
import pandas as pd
from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings

bs_faq = pd.read_csv('data/bookstore_faq.csv')
bs_eval = pd.read_csv('data/bookstore_eval.csv')

bs_documents = [Document(page_content=f'{r.title}\n{r.text}', metadata={'chunk_id': r.id})
                for r in bs_faq.itertuples()]
bs_embeddings = HuggingFaceEmbeddings(model_name='jhgan/ko-sroberta-multitask')
bs_store = Chroma.from_documents(bs_documents, bs_embeddings,
                                 collection_name='bookstore_eval', ids=list(bs_faq['id']))


def search_ids(query, k):
    """질문과 가장 가까운 FAQ k개의 id 를 순위 순서로 돌려준다."""
    return [d.metadata['chunk_id']
            for d in bs_store.as_retriever(search_kwargs={'k': k}).invoke(query)]


print('FAQ', len(bs_faq), '건 색인 / 평가셋', len(bs_eval), '문항')
display(bs_eval.head(3))

### 1단계 — 지표 네 개를 다시 준비

LV1 9~12번에서 만든 **`hit_at_k`·`precision_at_k`·`recall_at_k`·`mrr_at_k`** 를 이 노트북에도 정의하세요. 인자는 그때와 같습니다 — `(ranked, gold, k)` 이고 넷 다 **실수**를 돌려줍니다.

**확인**: `hit_at_k(['a', 'b'], ['b'], 2)` 가 `1.0`, `precision_at_k(['a', 'b'], ['b'], 2)` 가 `0.5`, `recall_at_k(['a', 'b'], ['b', 'z'], 2)` 가 `0.5`, `mrr_at_k(['a', 'b'], ['b'], 2)` 가 `0.5` 입니다.

<details><summary>힌트</summary>

```text
접근방법:
- LV1 에서 만든 네 함수를 그대로 다시 쓴다. 새로 배우는 것은 없다.

세부구현:
1. 상위 k개를 자르고 정답 목록과 견주는 방식은 넷 다 같다.
2. 무엇으로 나누는지만 다르다 - k 로 나누면 정밀도, 정답 개수로 나누면 재현율이다.
3. 순위가 필요한 것은 하나뿐이다.
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert hit_at_k(['a', 'b'], ['b'], 2) == 1.0 and hit_at_k(['a', 'b'], ['z'], 2) == 0.0
assert abs(precision_at_k(['a', 'b'], ['b'], 2) - 0.5) < 1e-9
assert abs(recall_at_k(['a', 'b'], ['b', 'z'], 2) - 0.5) < 1e-9
assert abs(mrr_at_k(['a', 'b'], ['b'], 2) - 0.5) < 1e-9
assert mrr_at_k(['a', 'b'], ['z'], 2) == 0.0, '상위 k개 안에 정답이 없으면 0.0 입니다'
print('✅ 통과!')

### 2단계 — K 별로 재서 표 만들기

`K` 를 **1·2·3·5** 로 바꿔 가며 평가셋 **전체 평균**을 재고, 그 결과를 DataFrame **`k_table`** 에 담으세요.

- 정답 라벨은 `bs_eval` 의 **`gold_chunks`** 열에 `'|'` 로 이어져 있습니다 — 나눠서 리스트로 쓰세요.
- 검색은 제공된 **`search_ids(query, k)`** 를 쓰세요.
- `k_table` 의 열은 **`['K', 'Hit', 'P', 'R', 'MRR']`** 이고 행은 K 오름차순 **4행**입니다.
- 각 지표 값은 그 K 로 잰 **14문항 평균**입니다.

**예시**: `k_table` 의 첫 행은 `K=1` 이고, 그때 `Hit` 은 0.9 보다 작습니다.

<details><summary>힌트</summary>

```text
접근방법:
- 바깥 반복은 K, 안쪽 반복은 문항이다. 문항마다 한 번만 검색해 네 지표를 모두 뽑는다.

세부구현:
1. K 값 네 개를 순서대로 돈다.
2. 그 안에서 평가셋의 행을 돌며
   2-1. 질문으로 상위 K개 id 를 찾고
   2-2. 정답 문자열을 구분자로 나눠 리스트로 만들고
   2-3. 네 지표를 각각 계산해 모아 둔다.
3. 네 지표의 평균을 그 K 의 한 행으로 담는다.
4. 행들을 DataFrame 으로 만든다.
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert list(k_table.columns) == ['K', 'Hit', 'P', 'R', 'MRR'], '열 이름과 순서를 맞춰 주세요'
assert list(k_table['K']) == [1, 2, 3, 5]
# 재현율은 K 가 커질수록 줄어들 수 없고, 정밀도는 커질수록 늘어날 수 없다(구조적 성질)
assert list(k_table['R']) == sorted(k_table['R'])
assert list(k_table['P']) == sorted(k_table['P'], reverse=True)
# 실측값과 대조 - 검색은 결정적이라 같은 색인에서 같은 값이 나온다
assert abs(k_table.loc[0, 'Hit'] - 0.857) < 0.02, 'K=1 의 Hit 이 실측과 다릅니다'
assert abs(k_table.loc[0, 'R'] - 0.643) < 0.02, 'K=1 의 Recall 이 실측과 다릅니다'
assert abs(k_table.loc[1, 'R'] - 0.929) < 0.02, 'K=2 의 Recall 이 실측과 다릅니다'
assert abs(k_table.loc[3, 'P'] - 0.271) < 0.02, 'K=5 의 Precision 이 실측과 다릅니다'
print('✅ 통과!')

### 3단계 — 규칙에 따라 `k` 를 고르기

이제 정합니다. 우리 서비스의 기준은 이렇습니다.

> **답에 필요한 근거를 되도록 다 건지되(재현율 0.9 이상), 관련 없는 글은 되도록 적게 넣는다.**

- 이 기준을 만족하는 **가장 작은 K** 를 변수 **`chosen_k`** 에 **정수**로 담으세요.
- 왜 그 값인지 근거가 되는 두 수 — 그 K 의 재현율과 정밀도 — 를 각각 **`chosen_recall`**, **`chosen_precision`** 에 담으세요(실수).
- 세 값 모두 `k_table` 에서 **찾아내야** 합니다. 눈으로 보고 손으로 적으면 안 됩니다.

<details><summary>힌트</summary>

```text
접근방법:
- 조건을 만족하는 행만 남기고, 그중 K 가 가장 작은 행을 고른다.

세부구현:
1. 재현율이 기준 이상인 행만 걸러 낸다.
2. 남은 행을 K 오름차순으로 보고 첫 행을 고른다.
3. 그 행에서 K 와 두 지표를 꺼내 각각 담는다.
   3-1. K 는 정수여야 한다.
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert isinstance(chosen_k, int), 'chosen_k 는 정수여야 합니다'
assert chosen_k == 2, '기준을 만족하는 가장 작은 K 를 다시 확인하세요'
# 손으로 적은 값이 아니라 k_table 에서 꺼낸 값인지 대조한다
_row = k_table[k_table['K'] == chosen_k].iloc[0]
assert abs(chosen_recall - _row['R']) < 1e-9, 'chosen_recall 은 k_table 에서 꺼내세요'
assert abs(chosen_precision - _row['P']) < 1e-9, 'chosen_precision 은 k_table 에서 꺼내세요'
assert chosen_recall >= 0.9, '고른 K 가 기준을 만족하지 않습니다'
# 더 작은 K 는 기준을 만족하지 않아야 한다(가장 작은 K 여야 하므로)
_smaller = k_table[k_table['K'] < chosen_k]
assert (_smaller['R'] < 0.9).all(), '더 작은 K 로도 기준을 만족합니다 - 다시 고르세요'
print('✅ 통과!')

### 4단계 — 못 찾은 문항 읽기 (서술형)

평균만 보고 끝내지 않습니다. **`chosen_k`** 로 쟀을 때 **`Hit` 이 0 인 문항**을 찾아 출력하세요 — 질문·정답 라벨·실제 검색 결과를 나란히 봅니다. 변수 이름은 **`missed`**(문항 id 의 리스트)로 하세요.

그 문항을 실제로 읽고, **아래 markdown 셀에** 두 가지를 적으세요.

1. 왜 못 찾았다고 생각하나요? (질문의 낱말과 정답 FAQ 의 낱말을 견주어 보세요)
2. `K` 를 더 키우면 이 문항이 해결될까요? 표를 근거로 답하세요.

<details><summary>힌트</summary>

```text
접근방법:
- 문항마다 고른 K 로 검색해 정답이 하나도 안 들어왔는지 본다.

세부구현:
1. 평가셋을 돌며 고른 K 로 검색한다.
2. 정답 목록과 겹치는 것이 하나도 없으면 그 문항 id 를 모은다.
3. 모은 문항의 질문·정답·검색 결과를 차례로 출력한다.
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert isinstance(missed, list) and len(missed) == 1, '고른 K 에서 못 찾은 문항은 하나입니다'
# missed 가 손으로 적은 값이 아니라 실제로 재서 나온 값인지 대조한다
_want = [r.query_id for r in bs_eval.itertuples()
         if hit_at_k(search_ids(r.query, chosen_k), r.gold_chunks.split('|'), chosen_k) == 0.0]
assert missed == _want, 'missed 는 실제로 재서 모은 목록이어야 합니다'
print('✅ 통과!')

**답안** *(아래에 두 물음의 답을 서술하세요)*

*(여기에 자신의 판단을 서술하세요)*

---
수고했어요! 흩어진 리뷰를 **구조화 수집 → 집계 → 인사이트**로 잇는 파이프라인을 완성했고, 검색기를 **재서 `k` 를 근거 있게 고르는** 일까지 해 봤습니다. 특히 마지막 문제는 모델을 한 번도 부르지 않았지요 — **판단의 근거를 수치로 만드는 일**에는 모델이 필요 없습니다. 다음 단원 **ReAct 에이전트** 에서는 에이전트가 도구를 고르는 **사고 루프의 원리**를 직접 해부합니다.